# Azure OpenAI를 활용한 Chat Completions

이 노트북에서는 **Microsoft Foundry**와 **`azure-ai-inference`** SDK를 사용해 Chat Completions를 수행하는 방법을 설명합니다.

1. `ChatCompletionsClient`를 초기화합니다.
2. 시스템 컨텍스트를 추가하기 위해 프롬프트 템플릿을 사용합니다.
3. 건강/피트니스 주제의 사용자 프롬프트를 전송합니다.

## 건강-피트니스 관련 안내
> 이 예제는 데모 목적일 뿐이며 실제 의학적 조언을 제공하지 않습니다.
> 건강이나 의학 관련 질문은 반드시 전문가와 상담하세요.

### 사전 준비 사항
이 노트북을 시작하기 전에 루트 `README.md`에 명시된 사전 조건을 완료했는지 확인하세요.

## 1. 초기 설정  
환경 변수를 로드하고, `ChatCompletionsClient`를 생성합니다.
또한 시스템 메시지를 구성하는 방식을 보여주기 위해 **프롬프트 템플릿**도 정의할 예정입니다.


In [ ]:
# 이 노트북에서 필요한 패키지 설치 (커널당 1회)

%pip install -q python-dotenv azure-ai-inference azure-core


In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage

# 환경 변수 로드
def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip('"').strip("'")

def _derive_models_endpoint(project_endpoint: str | None) -> str | None:
    if not project_endpoint:
        return None
    parsed = urlparse(project_endpoint)
    if not parsed.scheme or not parsed.netloc:
        return None
    return f"{parsed.scheme}://{parsed.netloc}/models"

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ .env 로드 완료: {dotenv_path}")
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 현재/상위 폴더를 확인하세요.")

# PROJECT_*만 입력하면 endpoint/key를 구성합니다.
project_endpoint = _clean_env(os.environ.get("PROJECT_ENDPOINT"))
project_api_key = _clean_env(os.environ.get("PROJECT_API_KEY"))
endpoint = _derive_models_endpoint(project_endpoint)
api_key = project_api_key
chat_model = _clean_env(os.environ.get("MODEL_NAME"))

if endpoint:
    print("ℹ️ Models endpoint 자동 구성 완료: <host>/models")

chat_client = None
missing_keys = [
    name
    for name, value in {
        "PROJECT_ENDPOINT": project_endpoint,
        "PROJECT_API_KEY": api_key,
        "MODEL_NAME": chat_model,
    }.items()
    if not value
]
if missing_keys:
    print(f"❌ 필수 환경 변수가 없습니다: {', '.join(missing_keys)}")
else:
    try:
        chat_client = ChatCompletionsClient(
            endpoint=endpoint,
            credential=AzureKeyCredential(api_key),
            model=chat_model,
        )
        print("✅ ChatCompletionsClient 생성 완료")
    except Exception as e:
        print("❌ 클라이언트 초기화 오류:", e)

### 프롬프트 템플릿  

한국 유저 응대에 맞춘 시스템 메시지 템플릿 예시입니다.

시스템 프롬프트 (한국어 템플릿):

```
당신은 FitChat Korea, 친절한 한국어 피트니스 어시스턴트입니다.
항상 공손한 존댓말로 답변하고, 초보자도 이해하기 쉽게 단계별로 안내하세요.
운동 루틴을 제안할 때는 빈도, 강도, 휴식, 주의사항을 간단히 포함하세요.
매 답변에 다음 안내를 자연스럽게 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요.
```

그런 다음 사용자 입력을 `user` 메시지로 전달합니다.


In [ ]:
# 시스템 프롬프트 + 사용자 프롬프트로 Chat Completions를 호출하는 함수
def chat_with_fitness_assistant(user_input: str):
    """한국어 시스템 지침을 적용해 모델 응답을 반환합니다."""
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check PROJECT_ENDPOINT/PROJECT_API_KEY/MODEL_NAME in .env")

    system_text = (
        "당신은 FitChat Korea, 친절한 한국어 피트니스 어시스턴트입니다.\n"
        "항상 공손한 존댓말로 답변하고, 초보자도 이해하기 쉽게 단계별로 안내하세요.\n"
        "운동 루틴을 제안할 때는 빈도, 강도, 휴식, 주의사항을 간단히 포함하세요.\n"
        "매 답변에 다음 안내를 자연스럽게 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요."
    )

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_text),
            UserMessage(content=user_input)
        ],
    )

    return response.choices[0].message.content

print("한국어 로컬라이징된 helper function을 정의했습니다.")

## 2. Chat Completions 실행해보기 
건강이나 피트니스에 관한 사용자의 질문으로 함수를 호출하고, 결과를 확인해보겠습니다.  
질문을 자유롭게 수정하거나 여러 번 실행해보셔도 좋습니다!


In [ ]:
user_question = "집에서 초보자 운동 루틴을 시작하려면 어떻게 해야 할까요?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ 사용자 질문:", user_question)
print("🤖 어시스턴트 답변:", reply)

## 3. 채우기 형식의 프롬프트 템플릿

시스템 메시지에 변수를 추가해 사용자별 맞춤 응답을 구성할 수 있습니다.  

```
당신은 FitChat Korea, {name}님을 위한 한국어 AI 피트니스 코치입니다.
사용자의 목표는 다음과 같습니다: {goal}.
답변은 공손한 존댓말로 작성하고, 실행 가능한 3~5단계 행동 계획으로 제시하세요.
마지막에는 반드시 다음 문구를 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요.
```


In [ ]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check PROJECT_ENDPOINT/PROJECT_API_KEY/MODEL_NAME in .env")

    # 사용자별 값을 채워 넣는 시스템 템플릿
    system_template = (
        "당신은 FitChat Korea, {name}님을 위한 한국어 AI 피트니스 코치입니다.\n"
        "사용자의 목표는 다음과 같습니다: {goal}.\n"
        "답변은 공손한 존댓말로 작성하고, 실행 가능한 3~5단계 행동 계획으로 제시하세요.\n"
        "마지막에는 반드시 다음 문구를 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요."
    )

    system_prompt = system_template.format(name=user_name, goal=goal)
    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_input)
        ],
    )
    return response.choices[0].message.content

# 템플릿 예시 실행
templated_user_input = "평일에 시간이 거의 없는데, 집에서 20분 안에 할 수 있는 운동 루틴을 추천해 주세요."
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="지훈",
    goal="체지방 감량과 기초 체력 향상"
)
print("🗣️ 사용자 질문:", templated_user_input)
print("🤖 어시스턴트 답변:", assistant_reply)